# Ch.1 — GPU Architecture Fundamentals

**No GPU required.** Every cell in this notebook runs on any laptop using NumPy, Pandas, and Matplotlib. You are building calculators and visualisers — the same tools a Platform Engineer uses to reason about hardware *before* spending money on it.

**What this notebook builds:**
1. A GPU spec database — compare cards side-by-side on the numbers that actually matter
2. A Roofline Model visualiser — see exactly where any operation sits relative to a GPU's limits
3. An arithmetic intensity calculator — compute the FLOP/byte ratio for every major LLM operation
4. A decode step simulator — trace one token generation through the hardware at the byte level
5. A batching curve — watch arithmetic intensity rise as batch size grows
6. An InferenceBase decision tool — which GPU should the startup buy?

In [ ]:
# TODO: Implement this cell
#  (Dependencies)
#
# Steps:
# 1. Dependencies
# 2. Plot results -- call `update()`
# 3. Process data
#
# Hint:
#    # implement using the APIs described above

## 1 · GPU Spec Database

The five numbers that matter for AI workloads — everything else is marketing.

| Spec | Why it matters |
|---|---|
| `bf16_tflops` | Peak compute for LLM training/inference (use this, not FP32) |
| `bandwidth_tbs` | HBM bandwidth in TB/s — the real bottleneck for decode |
| `vram_gb` | Maximum model + KV cache you can fit |
| `nvlink_bw_gbs` | Intra-node GPU-to-GPU bandwidth (0 = PCIe only) |
| `tdp_w` | Watts — cooling, rack density, electricity cost |

Run the cell below to load the database, then explore `gpu_db`.

In [ ]:
# TODO: Implement this cell
#  (GPU Spec Database)
#
# Steps:
# 1. Set up: GPU Spec Database
# 2. Compute `gpu_specs` using `note()`
# 3. Compute `cols` using `DataFrame()`
# 4. Compute `gpu_db["Ridge_Point_FLOPbyte"]`
# 5. Call `to_string()` to produce the result
#
# Hint:
#    gpu_db = pd.DataFrame(???)

In [ ]:
# TODO: Implement this cell
#  (Bar chart: bandwidth per dollar vs TFLOPS per dollar)
#
# Steps:
# 1. Set up: Bar chart: bandwidth per dollar vs TFLOPS per dollar
# 2. Plot results -- call `subplots()`
# 3. Compute `order` using `sort_values()`
# 4. Plot results -- call `barh()`
# 5. Plot results -- call `sort_values()`
# 6. Plot results -- call `Patch()`
# 7. Call `scale()` to produce the result
#
# Hint:
#    axes = plt.subplots(???)
#    order = gpu_db.sort_values(???)
#    bars = ax.barh(???)
#    order2 = gpu_db.sort_values(???)

## 2 · The Roofline Model

The Roofline Model answers one question: *for a given GPU, is my operation limited by compute or by memory?*

$$\text{Achievable TFLOP/s} = \min\!\left(\text{Peak TFLOP/s},\;\; I \times \text{Bandwidth (TB/s)}\right)$$

Where $I$ is **arithmetic intensity** in FLOP/byte:

$$I = \frac{\text{FLOPs performed}}{\text{Bytes read from HBM}}$$

- If $I < I_\text{ridge}$: compute units idle waiting for data → **memory-bound**
- If $I > I_\text{ridge}$: memory system is idle → **compute-bound**
- $I_\text{ridge} = \text{Peak TFLOP/s} \div \text{Bandwidth (TB/s)}$

The `plot_roofline()` function below works for any GPU in `gpu_db`. Pass a list of GPU names and optionally a list of `(label, intensity)` operation points to overlay.

In [ ]:
def plot_roofline(gpu_names, operations=None, ax=None, title=None):
    """
    TODO #4: Implement `plot_roofline()`.

    Steps:
    1. Set up: Roofline Model: reusable plotter
    2. Define helper function `plot_roofline()`
    3. Call `logspace()` to produce the result
    4. Process data
    5. Plot results -- call `minimum()`
    6. Plot results -- call `points()`
    7. Plot results -- call `text()`
    8. Plot results -- call `subplots()`
    9. Call `points()` to produce the result

    Hint:
    ax = plt.subplots(???)
    intensity_range = np.logspace(???)
    achievable = np.minimum(???)
    op_colors = plt.cm.Set2(???)

    Returns: ax
    """
    raise NotImplementedError("TODO: implement plot_roofline()")

## 3 · Arithmetic Intensity of LLM Operations

Now let's calculate where real LLM operations land on that roofline.

For a matrix multiply $[B \times M] \times [M \times N]$:

$$\text{FLOPs} = 2 \cdot B \cdot M \cdot N$$
$$\text{Bytes} = (B \cdot M + M \cdot N + B \cdot N) \times \text{bytes per element}$$
$$I = \frac{2BMN}{(BM + MN + BN) \times \text{bpe}}$$

When $B = 1$ (single user, token-by-token decode), the weight matrix $[M \times N]$ completely dominates the byte count. The intensity collapses toward $\approx 1 / \text{bpe}$ — nearly zero relative to the ridge point. This is the root cause of why single-user LLM inference is so GPU-inefficient.

In [ ]:
def gemm_arithmetic_intensity(B, M, N, bytes_per_element=2):
    """
    TODO #5: Implement `gemm_arithmetic_intensity()`.

    Steps:
    1. Set up: Arithmetic intensity calculator for matrix multiplies
    2. Define helper function `gemm_arithmetic_intensity()`
    3. Llama-3-8B model dimensions
    4. Compute `A100_RIDGE`
    5. Compute `ops_b1` using `phase()`
    6. Call `1()` to produce the result

    Hint:
    label = op.replace(???)

    Returns: flops / bytes_hbm
    """
    raise NotImplementedError("TODO: implement gemm_arithmetic_intensity()")

In [ ]:
# TODO: Implement this cell
#  (Plot all LLM operations on the A100 roofline)
#
# Steps:
# 1. Set up: Plot all LLM operations on the A100 roofline
# 2. Compute `ops_for_plot` using `gemm_arithmetic_intensity()`
# 3. Compute `decode_ops`
# 4. Compute `intensity_range` using `logspace()`
# 5. Plot results -- call `subplots()`
# 6. Plot results -- call `items()`
# 7. Plot results -- call `Line2D()`
# 8. Call `ops()` to produce the result
#
# Hint:
#    intensity_range = np.logspace(???)
#    ax = plt.subplots(???)
#    achievable = np.minimum(???)

## 4 · Decode Step Simulator — One Token at the Hardware Level

Let's trace exactly what happens during a single decode step for Llama-3-8B: FLOPs performed, bytes moved from HBM, and the resulting minimum latency bounded by memory bandwidth — not compute.

The simulator accounts for every operation in every layer: QKV projections, attention score computation, softmax + weighted sum, output projection, and both FFN linear layers. The LM head (vocab projection) is included once.

In [ ]:
def decode_step_analysis(
    batch_size = 1,
    hidden_dim = 4096,
    num_layers = 32,
    num_heads  = 32,
    head_dim   = 128,
    ffn_dim    = 14336,
    vocab_size = 32000,
    seq_len    = 512,
    bpe        = 2,       # bytes per element (BF16):
    """
    TODO #7: Implement `decode_step_analysis()`.

    Steps:
    1. Set up: Full decode step: FLOP and HBM byte accounting for Llama-3-8B
    2. Define helper function `decode_step_analysis()`
    3. Process data
    4. Call `append()` to produce the result
    5. Call `DataFrame()` to produce the result
    6. Process data
    7. Single-user decode analysis
    8. Compute `result_b1`
    9. Call `GPU()` to produce the result
    10. Process data

    Hint:
    df = pd.DataFrame(???)
    total_flops = df.filter(???)
    total_bytes = df.filter(???)

    Returns: {
    """
    raise NotImplementedError("TODO: implement decode_step_analysis()")

In [ ]:
# TODO: Implement this cell
#  (Per-operation breakdown: FLOPs vs HBM bytes in one layer)
#
# Steps:
# 1. Set up: Per-operation breakdown: FLOPs vs HBM bytes in one layer
# 2. Compute `df`
# 3. Compute `avg_flops` using `mean()`
# 4. Plot results -- call `arange()`
# 5. Compute `bars1` using `get_height()`
# 6. Compute `bars2` using `get_height()`
# 7. Plot results -- call `suptitle()`
# 8. Call `matrices()` to produce the result
#
# Hint:
#    x = np.arange(???)
#    bars1 = ax1.bar(???)
#    bars2 = ax2.bar(???)

## 5 · The Batching Curve — Why Batching Transforms Efficiency

When processing $B$ tokens simultaneously instead of 1, the weight matrices are still the same size (loaded once from HBM), but you perform $B\times$ more FLOPs on them. Arithmetic intensity rises linearly with batch size — until you cross the ridge point and become compute-bound.

$$I(B) \xrightarrow{B \gg 1} \frac{2B}{\text{bpe}}$$

This is the fundamental reason why **continuous batching** (Ch.5) is the single most impactful inference optimisation: it raises the effective batch size from ~1 toward 32–128 without adding latency per user.

In [ ]:
# TODO: Implement this cell
#  (Batch size sweep: intensity and GPU utilisation on A100)
#
# Steps:
# 1. Set up: Batch size sweep: intensity and GPU utilisation on A100
# 2. Compute `batch_sizes` using `arange()`
# 3. Compute `a100`
# 4. Compute `achievable` using `minimum()`
# 5. Plot results -- call `subplots()`
# 6. Call `fill_between()` to produce the result
# 7. Compute `cross` using `axvline()`
# 8. Call `semilogx()` to produce the result
# 9. Call `scatter()` to produce the result
# 10. Plot results -- call `tight_layout()`
# 11. Call `numbers()` to produce the result
#
# Hint:
#    batch_sizes = np.arange(???)
#    intensities = np.array(???)
#    achievable = np.minimum(???)

## 6 · InferenceBase Decision Tool

The Platform Engineer needs to answer: **which GPU should InferenceBase buy to serve Llama-3-8B at 5,000 tokens/second total (100 concurrent users × 50 tok/s each), within a $15,000/month cloud budget?**

We model this using only bandwidth arithmetic and published cloud pricing — no benchmarks, no live GPU required.

In [ ]:
# TODO: Implement this cell
#  (InferenceBase: GPU selection model)
#
# Steps:
# 1. Set up: InferenceBase: GPU selection model
# 2. Compute `TARGET_TOKS_PER_SEC`
# 3. Compute `cloud_instances` using `prices()`
# 4. Compute `res_b32` using `decode_step_analysis()`
# 5. Call `step()` to produce the result
# 6. Compute `rows`
# 7. Call `items()` to produce the result
# 8. Call `append()` to produce the result
# 9. Call `GB()` to produce the result
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Visualise cost and throughput comparison)
#
# Steps:
# 1. Set up: Visualise cost and throughput comparison
# 2. Compute `results_df` using `DataFrame()`
# 3. Compute `bar_colors`
# 4. Plot results -- call `subplots()`
# 5. Compute `bars1` using `barh()`
# 6. Compute `bars2` using `barh()`
# 7. Plot results -- call `tight_layout()`
# 8. Compute `in_budget` using `pricing()`
#
# Hint:
#    results_df = pd.DataFrame(???)
#    gpus = results_df.index.tolist(???)
#    bars1 = ax1.barh(???)
#    bars2 = ax2.barh(???)

## 7 · Sensitivity Analysis — How Model Size Changes Everything

How do the key metrics change as the model scales from 1B to 70B parameters? The VRAM footprint is the hard constraint. Arithmetic intensity at batch=1 barely changes with model size — all models are equally memory-bound per token. But the bandwidth requirement (HBM bytes per decode step) scales directly with parameter count.

In [ ]:
# TODO: Implement this cell
#  (Model size sweep: VRAM, HBM traffic, arithmetic intensity)
#
# Steps:
# 1. Set up: Model size sweep: VRAM, HBM traffic, arithmetic intensity
# 2. Compute `model_configs`
# 3. Compute `sweep_rows` using `items()`
# 4. Compute `sweep_df` using `DataFrame()`
#
# Hint:
#    sweep_df = pd.DataFrame(???)

In [ ]:
# TODO: Implement this cell
#  (Plot model size sweep)
#
# Steps:
# 1. Set up: Plot model size sweep
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `BF16()`
# 4. Plot results -- call `intensity()`
# 5. Plot results -- call `step()`
# 6. Plot results -- call `suptitle()`
#
# Hint:
#    axes = plt.subplots(???)
#    labels = sweep_df.index.tolist(???)
#    bars = ax.bar(???)
#    bars3 = ax3.bar(???)

## Exercises

Work through these before moving to Ch.2. Each one extends the calculators above.

**Exercise 1** — Add your own GPU to `gpu_specs` in section 1 and re-run all cells. Where does it sit on the roofline?

**Exercise 2** — FP8 inference on H100. Re-run `decode_step_analysis` with `bpe=0.5`. Does the H100's higher ridge point make FP8 decode compute-bound before BF16?

**Exercise 3** — Longer context. Re-run `decode_step_analysis` with `seq_len=4096`. Does quadrupling the context significantly change arithmetic intensity?

**Exercise 4** — The 70B upgrade. Model what happens if InferenceBase upgrades to Llama-3-70B. Does any single GPU fit the model in BF16? What is the minimum GPU count and cost?

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `result_fp8` using `decode_step_analysis()`
# 2. Process data
# 3. Compute `b_arr` using `array()`
# 4. Plot results -- call `semilogx()`
# 5. Compute `cross_bf16`
#
# Hint:
#    b_arr = np.arange(???)
#    i_bf16 = np.array(???)
#    i_fp8 = np.array(???)
#    ax = plt.subplots(???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `short` using `decode_step_analysis()`
# 2. Call `context()` to produce the result
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `res_70b` using `decode_step_analysis()`
# 2. Call `analysis()` to produce the result
# 3. Call `items()` to produce the result
#
# Hint:
#    # implement using the APIs described above